In [ ]:
# Which task to run TSP on?
task = 1  # 1, 2, 3, 4

In [ ]:
# Importing everything that is necessary
import itertools
import json

import gurobipy as gp

from data_helpers import read_distance_matrix_csv
from plot_helpers import map_a_tour

In [ ]:
# Helper functions for the main code below


# Using lazy constraints add sub-tour elimination constraints
def subtour_elimination_callback(model, where):
    # If this is a callback, in which Gurobi has found a feasible solution
    if where == gp.GRB.Callback.MIPSOL:
        # Get the values in the current solution for the decision variables
        x = model.cbGetSolution(model._vars)
        # Find currently selected edges
        selected_edges = gp.tuplelist((i, j) for i, j in model._vars.keys() if x[i, j] > 0.5)
        # Find the shortest cycle in the selected edge list
        shortest_subtour = get_shortest_subtour(selected_edges)
        # If the sub-tour isn't full circle (TSP solution)
        if len(shortest_subtour) < n:
            # Add sub-tour elimination constraint for every pair of cities in the tour
            model.cbLazy(
                gp.quicksum(
                    model._vars[i, j] + model._vars[j, i] for i, j in itertools.combinations(shortest_subtour, 2)
                ) <= len(shortest_subtour)-1
            )


# Given a tuplelist of edges, find and return the shortest sub-tour as a list of nodes
def get_shortest_subtour(edges):
    # At first, we haven't visited any locations
    unvisited = list(range(n))
    # Initial length has 1 more city than n, so in case there is a full circle with n
    #   locations, it will be shorter than this
    cycle = range(n + 1)
    # While there is an unvisited location
    while unvisited:
        # Let's find a cycle
        current_cycle = []
        # We can only look at the locations that aren't visited
        neighbors = unvisited
        # While there are locations to look at
        while neighbors:
            # Take the first location
            current = neighbors[0]
            # Add it to the current cycle
            current_cycle.append(current)
            # Remove it from the unvisited list
            unvisited.remove(current)
            # Take all unvisited neighbors
            neighbors = [j for i, j in edges.select(current, "*") if j in unvisited]
        # If it's shorter than the current shortest cycle
        if len(current_cycle) < len(cycle):
            cycle = current_cycle
    return cycle


def read_credentials(file_path):
    with open(file_path, "r") as file:
        return json.load(file)

In [ ]:
# This is the real entry point of the program

# Reading credentials for your WLS license
credentials = read_credentials("credentials.json")

# Parsing them to the necessary format
params = {
    "WLSACCESSID": credentials["access_id"],
    "WLSSECRET": credentials["secret"],
    "LICENSEID": credentials["license_id"],
}

# Creating your own environment gives you more control over Gurobi licensing, and it is
#   necessary when running instances of the size we are looking at
env = gp.Env(params=params)

# Naming the model and specifying our environment
m = gp.Model(name="ATSP", env=env)

# Get the data for the problem
# dict is a dictionary of N*N items, where keys are from (0, 0) to (N - 1, N - 1) and the
#   values are, for example, 267.314
# addresses is a list of N addresses, where each item is in format (56.950559, 24.115600)
dist, addresses = read_distance_matrix_csv()

# Depending on the task chosen at the top, we need to limit the amount of data
if task == 1:
    lon_right_limit = 22.3
elif task == 2:
    lon_right_limit = 23.6
elif task == 3:
    lon_right_limit = 24.6
else:  # task == 4
    # Task 4 contains all locations and longitude of 30 is larger than for any of the
    #   locations
    lon_right_limit = 30.0

# Which addresses will be used?
use_locations = [1 if address[1] <= lon_right_limit else 0 for address in addresses]

# Find the indexes of addresses that will be used
indices = [i for i, x in enumerate(use_locations) if x == 1]  # This could also be above

# Limited distance dictionary
dist_limited = {}

for i, x in enumerate(indices):
    for j, y in enumerate(indices):
        dist_limited[(i, j)] = dist[(x, y)]

# Let's define the limited distances as all distances
dist = dist_limited

# All addresses that are used in our task
addresses = [address for i, address in enumerate(addresses) if use_locations[i] == 1]

# Setting n parameter based on amount of addresses used
n = len(addresses)
print(f"n: {n}")

# Let's define our decision variables
# If we do not tell GRB.BINARY specifically, it will allow values like 0.67, and we
#   can't use only 67% of a specific road
# x[i][j] = 1 means to use a road, 0 means to not use a road
# obj=dist sets variable coefficients as distances, if we don't set setObjective, as in
#   this case, it defaults to minimizing linear combination of coefficients
x = m.addVars(dist.keys(), obj=dist, vtype=gp.GRB.BINARY, name="edge")

# For each source location the sum should be 1, that is, only 1 destination location
#   should be supplied
m.addConstrs(x.sum(i, "*") == 1 for i in range(n))
# For each destination location the sum should be 1, that is, only 1 source location
#   should be supplied
m.addConstrs(x.sum("*", i) == 1 for i in range(n))
# For each location, there shouldn't be any roads chosen, where source and destination
#   locations are the same
m.addConstrs(x.sum(i, i) == 0 for i in range(n))

# Solve the model
m._vars = x
m.Params.lazyConstraints = 1
# Runs the optimization engine, periodically calling subtour_elimination_callback callback function
m.optimize(subtour_elimination_callback)

# Retrieve optimal solution of the TSP
# "x" to get the values in the current solution for the decision variables
vals = m.getAttr("x", x)
selected = gp.tuplelist((i, j) for i, j in vals.keys() if vals[i, j] > 0.5)

# Get the shortest sub-tour – for an optimal solution this is the final answer – tour
tour = get_shortest_subtour(selected)

# We need to append first location for the final tour, to have the full circle
tour.append(tour[0])

print(f"\nOptimal tour: {tour}")
print(f"Optimal cost: {m.objVal}")

# Output the structure of the model
m.write("2_ATSP.lp")

In [ ]:
# Map the solution
map_a_tour(tour, addresses)

In [ ]:
# Cleanup
m.dispose()  # Free all resources associated to this model
gp.disposeDefaultEnv()  # Disposes of default environment created by Gurobi